# Customer ETL

El notebook reutiliza el contrato, los controles de calidad y la transformación del job. No mantiene una segunda copia de la lógica de negocio.

In [ ]:
import sys
from pyspark.sql import SparkSession

sys.path.insert(0, "/opt/spark-apps/customer_etl/scripts")
from transforms import build_customer_loyalty, read_inputs, validate_inputs, validate_output

spark = SparkSession.builder.appName("CustomerETLNotebook").getOrCreate()

In [ ]:
orders, products, customers = read_inputs(
    spark, "file:///opt/spark-apps/landing/customer_etl"
)
input_metrics = validate_inputs(orders, products, customers)
input_metrics

In [ ]:
customer_loyalty = build_customer_loyalty(orders, products, customers)
output_metrics = validate_output(customer_loyalty, input_metrics["active_customers"])
customer_loyalty.show(truncate=False)
output_metrics

In [ ]:
# Preview opcional del laboratorio; el DAG publica una ruta aislada por RUN_DATE.
customer_loyalty.write.mode("overwrite").option("header", True).csv(
    "hdfs://hdfs-namenode:9000/customer_etl/output/notebook_preview"
)